In [6]:
# https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/data

In [7]:
import os
import time
import random
import pandas as pd
from faker import Faker

In [8]:
# INPUT_CSV = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-subset-10p.csv"
# OUTPUT_CSV = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-subset-10p-enriched.csv"
INPUT_CSV = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct.csv"
OUTPUT_CSV = "/home/lpascual/Projects/E-Commerce_DWH/data/bronze/landing/2019-Oct-enriched.csv"
CHUNK_SIZE = 5000

In [9]:
fake_user = Faker()

def parse_address(address:str):
    parts = {}
    components = address.split(", ")
    parts["city"] = components[1] if len(components) == 3 else components[1].split(" ")[0]
    state_zip = components[2] if len(components) == 3 else components[1].split(" ")
    parts["state_code"] = state_zip.split(" ")[0] if len(components) == 3 else state_zip[1]
    parts["zip_code"] = state_zip.split(" ")[1] if len(components) == 3 else state_zip[2]
    return parts

def generate_product_name():
    adjectives = ["Smart", "Eco", "Quantum", "Hyper", "Nano", "Bright", "Swift", "Silent", "Blue", "Turbo"]
    
    nouns = ["Pulse", "Core", "Drive", "Link", "Nest", "Forge", "Node", "Beam", "Spark", "Flow"]
    
    suffixes = ["", "", "", "X", "Pro", "One", "360", "Edge", "Max", "Lite"]

    name = f"{random.choice(adjectives)} {random.choice(nouns)} {random.choice(suffixes)}"
    return name

# Delete OUTPUT_CSV
if os.path.isfile(OUTPUT_CSV):
    os.remove(OUTPUT_CSV)
    print(f"{OUTPUT_CSV} has been removed successfully")

In [10]:
def enrich_row(row):
    fake_user.seed_instance(int(row["user_id"]))
    profile = fake_user.simple_profile()
    row["name"] = profile["name"]
    row["username"] = profile["username"]
    row["email"] = profile["mail"]
    row["address"] = profile["address"].replace("\n", ", ")
    address = parse_address(row["address"])
    row["city"] = address["city"]
    row["country"] = "US"
    row["state_code"] = address["state_code"]
    row["zip_code"] = address["zip_code"]
    row["sex"] = profile["sex"]
    row["product_name"] = generate_product_name()
    row["product_description"] = fake_user.sentence(nb_words=10)
    return row

# TODO: Maybe implement as a decorator
start_time = time.time()

# Process in chunks
with pd.read_csv(INPUT_CSV, chunksize=CHUNK_SIZE) as reader:
    for i, chunk in enumerate(reader):
        enriched_chunk = chunk.apply(enrich_row, axis=1)
        enriched_chunk.to_csv(
            OUTPUT_CSV,
            mode="a",
            index=False,
            header=(i == 0)  # Write header only once
        )

end_time = time.time()
print(f"Execution time: {end_time - start_time:.2f} seconds")

Execution time: 101938.47 seconds
